In [1]:
import sys
sys.path.append("..")
from benchmarks.guacamol.assess_goal_directed_generation import assess_goal_directed_from_smiles
import pandas as pd


conditions = {
    "alzheimer": ["AChE", "MAOB"],
    "schizophrenia": ["D2R", "_5HT2A"],
    "parkinson": ["D2R", "D3R"],
}

smiles_dict = {}

# Load generated molecules SMILES
for disease, targets in conditions.items():
    targets_file = "_".join(targets) + "_SUM.csv"
    smiles_list = pd.read_csv(f"../generated_molecules/25-epoch/{targets_file}")["SMILES"].tolist()
    smiles_dict[disease] = smiles_list

compo_gpt_results = assess_goal_directed_from_smiles(
    smiles_dict, json_output_file=None, benchmark_version="multitarget"
)
results_df = pd.DataFrame(compo_gpt_results["results"])
results_df["Approach"] = "CoMPO-GPT"
results_df

Running benchmark 1/3: alzheimer
alzheimer: 91 instead of 100
[WARNING] An incorrect number of distinct molecules was generated: 91 instead of 100. Padding scores with 9 zeros...
  Score: 0.738476
  Execution time: 0:00:00
Running benchmark 2/3: schizophrenia
schizophrenia: 86 instead of 100
[WARNING] An incorrect number of distinct molecules was generated: 86 instead of 100. Padding scores with 14 zeros...
  Score: 0.816442
  Execution time: 0:00:00
Running benchmark 3/3: parkinson
parkinson: 89 instead of 100
[WARNING] An incorrect number of distinct molecules was generated: 89 instead of 100. Padding scores with 11 zeros...
  Score: 0.821116
  Execution time: 0:00:00


,benchmark_name,score,optimized_molecules,execution_time,number_scoring_function_calls,metadata,Approach
0,alzheimer,0.738476,"[(O=C(CCN1CCCC1)c1ccc2c(c1)C(=O)c1ccccc1C2=O, ...",0,91,"{'top_1': 0.8107928184140997, 'top_10': 0.7810...",CoMPO-GPT
1,schizophrenia,0.816442,[(Clc1ccc2[nH]cc(CCN3CCN(c4cccc5c4OCCO5)CC3)c2...,0,86,"{'top_1': 0.8846498922436085, 'top_10': 0.8707...",CoMPO-GPT
2,parkinson,0.821116,"[(CN1CCN(C2Cc3ccccc3Sc3ccc(Cl)cc32)CC1, 0.8794...",0,89,"{'top_1': 0.8794492085541237, 'top_10': 0.8660...",CoMPO-GPT


In [4]:
from pathlib import Path
import numpy as np

# Ensure runtime warnings raise as errors to debug
import warnings
warnings.filterwarnings(
    action='error', message='',
    category=RuntimeWarning
)


METHODS = {"CoMPO-GPT": compo_gpt_results}

# Baselines
baselines = [
    "DeepLig",
    "POLYGON",
    "MTMol-GPT",
]

diseases = ["alzheimer", "schizophrenia", "parkinson"]

for baseline in baselines:
    print(baseline)
    # List of diseases to evaluate
    diseases = ["alzheimer", "schizophrenia", "parkinson"]
    
    # Load generated molecules for each disease
    smiles_dict = {}
    
    for disease in diseases:
        path = f"../generated_molecules/{baseline}/{disease}-mpo.csv"
        if Path(path).exists():
            df = pd.read_csv(path,header=None)
            smiles_dict[disease] = df[0].tolist()
        else:
            print(f"Warning: No file found for {baseline} - {disease}")

    # Run benchmarks and store results
    METHODS[baseline] = assess_goal_directed_from_smiles(
        smiles_dict=smiles_dict,
        json_output_file=None, 
        benchmark_version="multitarget"
    )

METHODS.keys()

DeepLig
Running benchmark 1/3: alzheimer
alzheimer: 48 instead of 100
[WARNING] An incorrect number of distinct molecules was generated: 48 instead of 100. Padding scores with 52 zeros...
  Score: 0.673877
  Execution time: 0:00:00
Running benchmark 2/3: schizophrenia
schizophrenia: 80 instead of 100
[WARNING] An incorrect number of distinct molecules was generated: 80 instead of 100. Padding scores with 20 zeros...
  Score: 0.787750
  Execution time: 0:00:00
Running benchmark 3/3: parkinson
parkinson: 38 instead of 100
[WARNING] An incorrect number of distinct molecules was generated: 38 instead of 100. Padding scores with 62 zeros...
  Score: 0.675166
  Execution time: 0:00:00
POLYGON
Running benchmark 1/3: alzheimer
  Score: 0.608869
  Execution time: 0:00:00
Running benchmark 2/3: schizophrenia
schizophrenia: 98 instead of 100
[WARNING] An incorrect number of distinct molecules was generated: 98 instead of 100. Padding scores with 2 zeros...
  Score: 0.687352
  Execution time: 0:00

dict_keys(['CoMPO-GPT', 'DeepLig', 'POLYGON', 'MTMol-GPT'])

In [5]:
metric_order = [
    "Score",
    "Target Response",
    "Blood-Brain Barrier",
    "CNS MPO",
    "Synthetic Accessibility",
]

# benchmark_order = [
#     "Alzheimer MPO",
#     "Schizophrenia MPO",
#     "Parkinson MPO",
# ]
benchmark_order = [
    "alzheimer",
    "schizophrenia",
    "parkinson",
]

report_results = list()

def get_metadata_keys(metadata):
    keys = list(metadata.keys())

    target = [k for k in keys if "GeometricMeanScoringFunction" in k]
    target_scores = metadata[target[0]] if len(target) > 0 else None

    bbb = [k for k in keys if "BBBResponseScoringFunction" in k]
    bbb_scores = metadata[bbb[0]] if len(bbb) > 0 else None

    sa = [k for k in keys if "SyntheticAccessibilityScoringFunction" in k]
    sa_scores = metadata[sa[0]] if len(sa) > 0 else None

    cns = [k for k in keys if "CNS_MPO_ScoringFunction" in k]
    cns_scores = metadata[cns[0]] if len(cns) > 0 else None

    return {
        "Target Response": target_scores,
        "Blood-Brain Barrier": bbb_scores,
        "Synthetic Accessibility": sa_scores,
        "CNS MPO": cns_scores        
    }


for method, report in METHODS.items():
    results = report["results"]
        
    for result in results:
        results_info = dict()

        results_info["Method"] = method
        results_info["Benchmark"] = result["benchmark_name"]
        results_info["Score"] = f"{result['score']:.5f}"

        metadata = result["metadata"]
        scores_dict = get_metadata_keys(metadata)

        for score_key in scores_dict:
            scores = scores_dict[score_key]
            if scores is None:
                continue

            mean_score = np.mean(scores)
            std_score = np.std(scores)

            results_info[score_key] = f"{mean_score:.3f} ± {std_score:.3f}"
        
        report_results.append(results_info.copy())

# Reorder the benchmarks
df = pd.DataFrame(report_results)   
df["Benchmark"] = pd.Categorical(df["Benchmark"], benchmark_order, ordered=True)
dfs = list()

for metric in metric_order:
    df_score = df.pivot_table(
        index=["Benchmark"],
        columns=["Method"],
        values=[metric],
        aggfunc=lambda x: x,
    )

    df_score.columns = df_score.columns.droplevel(0)
    df_score["Metric"] = metric

    dfs.append(df_score)

concat_dfs = pd.concat(dfs, axis=0).sort_values(by=["Benchmark"]).reset_index()

concat_dfs["Metric"] = pd.Categorical(
    concat_dfs["Metric"], categories=metric_order, ordered=True
)

# [["Benchmark", 'Metric'] + list(METHODS.keys())]
concat_dfs.set_index(["Benchmark", "Metric"])

Method                                     CoMPO-GPT        DeepLig   
Benchmark     Metric                                                  
alzheimer     Score                          0.73848        0.67388  \
              Target Response          0.505 ± 0.066  0.394 ± 0.087   
              Blood-Brain Barrier      0.651 ± 0.198  0.897 ± 0.111   
              CNS MPO                  0.867 ± 0.152  0.906 ± 0.060   
              Synthetic Accessibility  0.827 ± 0.065  0.941 ± 0.059   
schizophrenia Score                          0.81644        0.78775   
              Target Response          0.696 ± 0.091  0.556 ± 0.074   
              Blood-Brain Barrier      0.896 ± 0.109  0.888 ± 0.115   
              CNS MPO                  0.855 ± 0.083  0.910 ± 0.052   
              Synthetic Accessibility  0.809 ± 0.058  0.930 ± 0.046   
parkinson     Score                          0.82112        0.67517   
              Target Response          0.726 ± 0.090  0.548 ± 0.105   
              Blood-Brain Barrier      0.870 ± 0.146  0.917 ± 0.097   
              CNS MPO                  0.850 ± 0.077  0.893 ± 0.051   
              Synthetic Accessibility  0.806 ± 0.065  0.965 ± 0.031   

Method                                     MTMol-GPT        POLYGON  
Benchmark     Metric                                                 
alzheimer     Score                          0.74379        0.60887  
              Target Response          0.491 ± 0.070  0.607 ± 0.027  
              Blood-Brain Barrier      0.628 ± 0.227  0.342 ± 0.111  
              CNS MPO                  0.922 ± 0.119  0.514 ± 0.059  
              Synthetic Accessibility  0.830 ± 0.121  0.752 ± 0.009  
schizophrenia Score                          0.78914        0.68735  
              Target Response          0.570 ± 0.186  0.597 ± 0.022  
              Blood-Brain Barrier      0.683 ± 0.287  0.871 ± 0.018  
              CNS MPO                  0.869 ± 0.159  0.664 ± 0.003  
              Synthetic Accessibility  0.811 ± 0.110  0.607 ± 0.016  
parkinson     Score                          0.80239        0.69939  
              Target Response          0.602 ± 0.130  0.624 ± 0.034  
              Blood-Brain Barrier      0.749 ± 0.224  0.909 ± 0.071  
              CNS MPO                  0.890 ± 0.104  0.722 ± 0.020  
              Synthetic Accessibility  0.846 ± 0.090  0.476 ± 0.057